# Level 1: basic tests

Compares the blank agent to the curated agent on unseen questions, then runs three sneaky tests (ambiguous term, out of scope, unit trap). The Conversation API is about 5 questions per minute, so this notebook waits between calls.


## Bind config


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)

ids = spark.table(cfg.table("agent_ids")).collect()[0]
blank_id = ids["blank_space_id"]
curated_id = ids["curated_space_id"]
client = GenieClient(warehouse_id=cfg.warehouse_id)


## Ask both agents with a throttle

TIMEOUT, NO_ANSWER, and ERROR are FAIL. Answers are stored for the facilitator recap.


In [ ]:
import time
from shared.lib.genie_client import GenieClientError

QUESTIONS = [
    "What is average first pass yield as a percent by plant?",
    "How many operators work the Night shift?",
    "Which product type has the most scrap events?",
    "How many equipment feedback rows flag equipment as not ok?",
]
SNEAKY = [
    ("ambiguous", "How is the line doing?"),
    ("out_of_scope", "What was the CEO bonus last year?"),
    ("unit_trap", "Is OEE 85 or 0.85 in this data?"),
]

def ask(space_id: str, question: str) -> dict:
    try:
        started = client.start_conversation(space_id, question)
        conversation_id = started.get("conversation_id") or (started.get("conversation") or {}).get("id")
        message = started.get("message") or {}
        message_id = message.get("id") or started.get("message_id")
        done = client.wait_for_message(space_id, conversation_id, message_id, timeout_seconds=180)
        status = (done.get("status") or done.get("state") or "UNKNOWN").upper()
        if status in {"FAILED", "CANCELLED"}:
            return {"question": question, "status": "FAIL", "detail": status}
        if status != "COMPLETED":
            return {"question": question, "status": "FAIL", "detail": status or "TIMEOUT"}
        return {"question": question, "status": "COMPLETED", "detail": str(done)[:1500]}
    except GenieClientError as exc:
        text = str(exc)
        label = "TIMEOUT" if "TIMEOUT" in text else "ERROR"
        return {"question": question, "status": "FAIL", "detail": f"{label} {text[:400]}"}

records = []
for q in QUESTIONS:
    for label, sid in [("blank", blank_id), ("curated", curated_id)]:
        rec = ask(sid, q)
        rec["agent"] = label
        records.append(rec)
        print(label, rec["status"], q)
        time.sleep(13)
for kind, q in SNEAKY:
    rec = ask(curated_id, q)
    rec["agent"] = "curated_sneaky"
    rec["kind"] = kind
    records.append(rec)
    print("sneaky", kind, rec["status"], q)
    time.sleep(13)

spark.createDataFrame(records).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    cfg.table("basic_test_runs")
)
print("wrote", cfg.table("basic_test_runs"))
print("PASS notebook finished. Review statuses in the table. TIMEOUT/NO_ANSWER/ERROR are FAIL.")
